In [67]:
import pandas as pd
import numpy as np

In [68]:
de = pd.read_csv("Data/Context/Germany.csv")

In [69]:
de = de.drop(["Unnamed: 0"], axis=1)

In [70]:
de_2007_2015 = de.loc[6:15]

In [71]:
de_2007_2015

,TIME_PERIOD,ROL,GE,COC,RQ,VAA,CPI,FOP,JI,IUI,...,WTDA,EDBS,DBP,DBD,SSB,DWCP,PMSI,PT,EC,ELFE
6,2006,1.77,1.65,1.79,1.57,1.37,80.0,0.986,0.952,72.2,...,NaN,NaN,9.0,27.0,78.010595,79.591946,50.0,77.015617,76.735165,NaN
7,2007,1.76,1.63,1.73,1.62,1.34,78.0,0.986,0.952,75.2,...,5.38,NaN,9.0,27.0,78.044700,82.709673,50.0,77.197929,76.735165,NaN
8,2008,1.73,1.51,1.75,1.49,1.34,79.0,0.986,0.952,78.0,...,5.36,NaN,9.0,23.0,79.197611,82.821877,50.0,76.399686,76.735165,NaN
9,2009,1.65,1.58,1.75,1.52,1.33,80.0,0.986,0.952,79.0,...,5.18,NaN,9.0,22.5,79.363510,82.758552,50.0,76.198556,76.735165,NaN
10,2010,1.64,1.52,1.77,1.57,1.30,79.0,0.986,0.952,82.0,...,5.01,79.277568,9.0,22.5,79.567065,82.888440,50.0,78.957718,76.735165,NaN
11,2011,1.62,1.50,1.74,1.55,1.35,80.0,0.986,0.952,81.3,...,4.81,79.213356,9.0,17.5,80.744231,82.852483,50.0,76.428017,76.735165,NaN
12,2012,1.67,1.53,1.82,1.53,1.39,79.0,0.983,0.952,82.4,...,4.88,79.298046,9.0,17.5,80.867728,82.818229,50.0,76.817402,76.735165,NaN
13,2013,1.64,1.51,1.80,1.54,1.41,78.0,0.983,0.952,84.2,...,4.90,79.187647,9.0,17.5,80.918824,82.870361,50.0,79.063708,76.735165,NaN
14,2014,1.85,1.66,1.81,1.70,1.44,79.0,0.983,0.952,86.2,...,4.89,79.316127,9.0,17.5,80.997172,82.919183,62.0,76.843778,76.735165,NaN
15,2015,1.76,1.66,1.78,1.71,1.42,81.0,0.976,0.952,87.6,...,4.89,78.851145,9.0,17.5,80.627202,78.084024,62.0,77.020556,71.486986,10.1


In [72]:
def missing_values(df):
    for feature in df.columns:
        missing_values = df[feature].isna().sum()
        percentage = missing_values/len(df) * 100
        print(f'{feature} - {missing_values} - {round(percentage, 1)} %')

In [73]:
missing_values(de_2007_2015)

TIME_PERIOD - 0 - 0.0 %
ROL - 0 - 0.0 %
GE - 0 - 0.0 %
COC - 0 - 0.0 %
RQ - 0 - 0.0 %
VAA - 0 - 0.0 %
CPI - 0 - 0.0 %
FOP - 0 - 0.0 %
JI - 0 - 0.0 %
IUI - 0 - 0.0 %
CFI - 5 - 50.0 %
PR - 0 - 0.0 %
DI - 2 - 20.0 %
SOARS - 1 - 10.0 %
TOGP - 1 - 10.0 %
IPAB - 4 - 40.0 %
PTIP - 1 - 10.0 %
QOES - 1 - 10.0 %
AOLT - 1 - 10.0 %
FLTA - 1 - 10.0 %
EBOF - 1 - 10.0 %
FOWD - 1 - 10.0 %
QOMS - 1 - 10.0 %
CLER - 1 - 10.0 %
WTDA - 1 - 10.0 %
EDBS - 4 - 40.0 %
DBP - 0 - 0.0 %
DBD - 0 - 0.0 %
SSB - 0 - 0.0 %
DWCP - 0 - 0.0 %
PMSI - 0 - 0.0 %
PT - 0 - 0.0 %
EC - 0 - 0.0 %
ELFE - 9 - 90.0 %


In [74]:
de_2007_2015 = de_2007_2015.drop(["ELFE", "CFI", "IPAB", "EDBS"], axis=1)

In [75]:
def fill_with_mean(df):
    for feature in df.columns:
        mean = df[feature].mean()
        df[feature].fillna(value=mean, inplace=True)

In [76]:
fill_with_mean(de_2007_2015)

In [77]:
continuous_data = ['DBP', "DBD", "EDBS", "SSB", "DWCP", "PT", "EC", "IUI", "PMSI"]

In [78]:
def handle_outliers(df, continuous_features):

    print("Original Shape:", df.shape)
    
    outlier_summary = {}
    
    for feature in continuous_features:
        if feature in df.columns:
            Q1 = df[feature].quantile(0.25)
            Q3 = df[feature].quantile(0.75)
            IQR = Q3 - Q1
            lower_bound = Q1 - 1.5 * IQR
            upper_bound = Q3 + 1.5 * IQR
            
            outlier_mask = (df[feature] < lower_bound) | (df[feature] > upper_bound)
            num_outliers = outlier_mask.sum()
            
            if num_outliers > 0:
                non_outlier_mean = df.loc[~outlier_mask, feature].mean()
                
                df.loc[outlier_mask, feature] = non_outlier_mean
                
                print(f"{feature}: Found {num_outliers} outliers, replaced with mean = {non_outlier_mean:.2f}")
            else:
                print(f"{feature}: No outliers found")
    
    print("\nFinal Shape:", df.shape)
    print(f"Total features processed: {len(continuous_features)}")
    
    return df

In [79]:
de_2007_2015_no_o = handle_outliers(de_2007_2015, continuous_data)

Original Shape: (10, 30)
DBP: No outliers found
DBD: No outliers found
SSB: No outliers found
DWCP: Found 2 outliers, replaced with mean = 82.83
PT: Found 2 outliers, replaced with mean = 76.74
EC: Found 1 outliers, replaced with mean = 76.74
IUI: No outliers found
PMSI: Found 2 outliers, replaced with mean = 50.00

Final Shape: (10, 30)
Total features processed: 9


In [80]:
de_2007_2015_no_o

,TIME_PERIOD,ROL,GE,COC,RQ,VAA,CPI,FOP,JI,IUI,...,QOMS,CLER,WTDA,DBP,DBD,SSB,DWCP,PMSI,PT,EC
6,2006,1.77,1.65,1.79,1.57,1.37,80.0,0.986,0.952,72.2,...,5.017778,5.137778,5.033333,9.0,27.0,78.010595,82.829850,50.0,77.015617,76.735165
7,2007,1.76,1.63,1.73,1.62,1.34,78.0,0.986,0.952,75.2,...,5.110000,5.100000,5.380000,9.0,27.0,78.044700,82.709673,50.0,77.197929,76.735165
8,2008,1.73,1.51,1.75,1.49,1.34,79.0,0.986,0.952,78.0,...,5.140000,4.960000,5.360000,9.0,23.0,79.197611,82.821877,50.0,76.399686,76.735165
9,2009,1.65,1.58,1.75,1.52,1.33,80.0,0.986,0.952,79.0,...,4.960000,5.030000,5.180000,9.0,22.5,79.363510,82.758552,50.0,76.198556,76.735165
10,2010,1.64,1.52,1.77,1.57,1.30,79.0,0.986,0.952,82.0,...,4.930000,5.250000,5.010000,9.0,22.5,79.567065,82.888440,50.0,76.740193,76.735165
11,2011,1.62,1.50,1.74,1.55,1.35,80.0,0.986,0.952,81.3,...,4.780000,5.120000,4.810000,9.0,17.5,80.744231,82.852483,50.0,76.428017,76.735165
12,2012,1.67,1.53,1.82,1.53,1.39,79.0,0.983,0.952,82.4,...,4.950000,5.170000,4.880000,9.0,17.5,80.867728,82.818229,50.0,76.817402,76.735165
13,2013,1.64,1.51,1.80,1.54,1.41,78.0,0.983,0.952,84.2,...,5.100000,5.200000,4.900000,9.0,17.5,80.918824,82.870361,50.0,76.740193,76.735165
14,2014,1.85,1.66,1.81,1.70,1.44,79.0,0.983,0.952,86.2,...,4.980000,5.220000,4.890000,9.0,17.5,80.997172,82.919183,50.0,76.843778,76.735165
15,2015,1.76,1.66,1.78,1.71,1.42,81.0,0.976,0.952,87.6,...,5.210000,5.190000,4.890000,9.0,17.5,80.627202,82.829850,50.0,77.020556,76.735165


In [81]:
de_2007_2015_no_o.to_csv('Data/Context/Germany_2007_2015.csv')

In [82]:
de_2007_2015_no_o = de_2007_2015_no_o.drop(["TIME_PERIOD"], axis=1)

In [83]:
de_2007_2013_l = de_2007_2015_no_o.loc[13]

In [84]:
de_2007_2013_l

ROL       1.640000
GE        1.510000
COC       1.800000
RQ        1.540000
VAA       1.410000
CPI      78.000000
FOP       0.983000
JI        0.952000
IUI      84.200000
PR       90.000000
DI        8.310000
SOARS     5.540000
TOGP      4.980000
PTIP      4.390000
QOES      5.140000
AOLT      6.290000
FLTA      5.770000
EBOF      5.670000
FOWD      3.340000
QOMS      5.100000
CLER      5.200000
WTDA      4.900000
DBP       9.000000
DBD      17.500000
SSB      80.918824
DWCP     82.870361
PMSI     50.000000
PT       76.740193
EC       76.735165
Name: 13, dtype: float64

In [85]:
de_2007_2013_l.to_csv('Data/Context/Germany_2007_2013_last.csv')

In [20]:
de_2016_2022 = de.loc[16:22]

In [21]:

missing_values(de_2016_2022)

TIME_PERIOD - 0 - 0.0 %
ROL - 0 - 0.0 %
GE - 0 - 0.0 %
COC - 0 - 0.0 %
RQ - 0 - 0.0 %
VAA - 0 - 0.0 %
CPI - 0 - 0.0 %
FOP - 0 - 0.0 %
JI - 0 - 0.0 %
IUI - 0 - 0.0 %
CFI - 0 - 0.0 %
PR - 0 - 0.0 %
DI - 0 - 0.0 %
SOARS - 5 - 71.4 %
TOGP - 5 - 71.4 %
IPAB - 5 - 71.4 %
PTIP - 5 - 71.4 %
QOES - 5 - 71.4 %
AOLT - 5 - 71.4 %
FLTA - 5 - 71.4 %
EBOF - 5 - 71.4 %
FOWD - 5 - 71.4 %
QOMS - 5 - 71.4 %
CLER - 5 - 71.4 %
WTDA - 6 - 85.7 %
EDBS - 2 - 28.6 %
DBP - 2 - 28.6 %
DBD - 2 - 28.6 %
SSB - 2 - 28.6 %
DWCP - 2 - 28.6 %
PMSI - 2 - 28.6 %
PT - 2 - 28.6 %
EC - 2 - 28.6 %
ELFE - 0 - 0.0 %


In [22]:
de_2016_2022 = de_2016_2022.drop(["SOARS", "TOGP", "IPAB", "PTIP", "QOES", "AOLT", "FLTA", "EBOF",
                                 "FOWD", "QOMS", "CLER", "WTDA"], axis=1)

In [23]:
fill_with_mean(de_2016_2022)

In [24]:
continuous_data = ['DBP', "DBD", "EDBS", "SSB", "DWCP", "PT", "EC", "IUI", "PMSI"]

In [25]:
de_2016_2022_no_o = handle_outliers(de_2016_2022, continuous_data)

Original Shape: (7, 22)
DBP: No outliers found
DBD: Found 1 outliers, replaced with mean = 13.03
EDBS: Found 1 outliers, replaced with mean = 79.18
SSB: Found 2 outliers, replaced with mean = 82.24
DWCP: Found 1 outliers, replaced with mean = 78.14
PT: Found 1 outliers, replaced with mean = 82.16
EC: Found 1 outliers, replaced with mean = 70.77
IUI: No outliers found
PMSI: No outliers found

Final Shape: (7, 22)
Total features processed: 9


In [26]:
de_2016_2022_no_o.to_csv('Data/Context/Germany_2016_2022.csv')

In [27]:
de_2016_2022_no_o = de_2016_2022_no_o.drop(["TIME_PERIOD"], axis=1)

In [33]:
de_2016_2022_no_o

,ROL,GE,COC,RQ,VAA,CPI,FOP,JI,IUI,CFI,...,DI,EDBS,DBP,DBD,SSB,DWCP,PMSI,PT,EC,ELFE
16,1.58,1.66,1.78,1.81,1.36,81.0,0.976,0.952,84.2,57.9,...,8.63,79.425123,9.0,13.033333,82.244290,78.113460,62.0,82.167297,70.940538,10.3
17,1.57,1.61,1.80,1.78,1.43,81.0,0.971,0.919,84.4,58.4,...,8.61,79.428673,9.0,13.000000,82.202748,78.144349,62.0,82.157760,70.394090,10.1
18,1.59,1.52,1.90,1.76,1.44,80.0,0.973,0.919,87.0,58.0,...,8.68,79.023990,9.0,13.000000,82.214093,78.114656,60.0,82.140949,70.394090,10.3
19,1.58,1.50,1.87,1.71,1.35,80.0,0.973,0.919,88.1,58.2,...,8.68,79.022313,9.0,13.000000,82.338520,78.161160,60.0,82.105637,70.394090,10.3
20,1.52,1.31,1.83,1.58,1.37,80.0,0.965,0.919,89.8,56.5,...,8.67,79.180230,9.0,13.000000,82.244290,78.137981,60.0,82.155389,70.768508,10.1
21,1.57,1.29,1.78,1.63,1.42,80.0,0.965,0.919,91.4,57.3,...,8.67,79.090639,9.0,13.100000,82.233046,78.147131,60.8,82.180346,71.244120,12.3
22,1.53,1.29,1.82,1.52,1.41,79.0,0.952,0.919,91.6,57.2,...,8.80,79.090639,9.0,13.100000,82.233046,78.147131,60.8,82.180346,71.244120,12.5


In [29]:
de_2014_2020_l = de_2016_2022_no_o.loc[20]

In [30]:
de_2014_2020_l

ROL      1.520000
GE       1.310000
COC      1.830000
RQ       1.580000
VAA      1.370000
CPI     80.000000
FOP      0.965000
JI       0.919000
IUI     89.800000
CFI     56.500000
PR      81.000000
DI       8.670000
EDBS    79.180230
DBP      9.000000
DBD     13.000000
SSB     82.244290
DWCP    78.137981
PMSI    60.000000
PT      82.155389
EC      70.768508
ELFE    10.100000
Name: 20, dtype: float64

In [31]:
de_2014_2020_l.to_csv('Data/Context/Germany_2014_2020_last.csv')

In [34]:
de_2016_2022_no_o.columns.equals(de_2007_2015_no_o)

False